# 03 — GLO Temporal Feature Engineering Pipeline

**Project:** GlacierGuard-AI  
**Task:** Next-Year Glacial-Lake Area Change Prediction  
**Source Data:** Annual GLO GeoPackages (`raw/GLO/S1_20172024_NTB_GLOID_v1.02.gpkg` and `raw/GLO/S2_20172024_NTB_GLOID_v1.02.gpkg`)  
**Primary Research Target:** `NEXT_AREA_CHANGE = NEXT_AREA - AREA` (where `NEXT_YEAR - AREA_YEAR == 1`)  

---

### Architectural Principles
1. **Zero Future Leakage:** Target transition parameters (`NEXT_AREA`, `NEXT_YEAR`, `NEXT_YEAR_GAP`) and whole-period trend metrics (`EXPANSION_RATE`) are strictly excluded from model feature sets.
2. **Strict Temporal Continuity:** Backward lags and forward targets require consecutive observations (`YEAR_GAP == 1`). Multi-year observation gaps are never treated as 1-year changes.
3. **No Zero-Imputation:** Early unobserved historical lags remain `NULL`.
4. **Sensor Separation:** S1 and S2 feature matrices (`gold_s1`, `gold_s2`) are preserved as distinct Spark DataFrames for baseline comparisons.
5. **Chronological Splitting:** Training (2017–2021), Validation (2022), and Testing (2023) respect temporal precedence without random leakage.

## 1 — Environment & Spark Verification

In [1]:
# == 1.1  Verify Spark session via Databricks Connect ==
import os
os.environ.setdefault('DATABRICKS_SERVERLESS_COMPUTE_ID', 'auto')

try:
    _ = spark  # noqa: F821
    print('Spark session: available (Databricks cluster)')
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
    print('Spark session: created via Databricks Connect')

print(f'Spark version : {spark.version}')
print('Connected via : Databricks Connect (Spark Connect v2)')


Spark session: created via Databricks Connect
Spark version : 4.2.0
Connected via : Databricks Connect (Spark Connect v2)


In [2]:
# == 1.2  Library Imports ==
import pathlib
import geopandas as gpd
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, BooleanType
)

print(f'GeoPandas : {gpd.__version__}')
print(f'Pandas    : {pd.__version__}')
print(f'NumPy     : {np.__version__}')
print(f'PySpark   : {spark.version}')


GeoPandas : 1.1.4
Pandas    : 2.2.3
NumPy     : 2.1.3
PySpark   : 4.2.0


## 2 — Configuration & Constants

In [3]:
# == 2.1  Pipeline Configuration Constants ==
GLO_DIR_candidates = [
    pathlib.Path('../raw/GLO'),
    pathlib.Path('./raw/GLO'),
    pathlib.Path('/Workspace/raw/GLO'),
]

GLO_DIR = None
for p in GLO_DIR_candidates:
    if p.exists():
        GLO_DIR = p.resolve()
        break

if GLO_DIR is None:
    raise FileNotFoundError('Cannot locate raw/GLO directory.')

S1_ANNUAL_PATH = GLO_DIR / 'S1_20172024_NTB_GLOID_v1.02.gpkg'
S2_ANNUAL_PATH = GLO_DIR / 'S2_20172024_NTB_GLOID_v1.02.gpkg'

TARGET_COL = 'NEXT_AREA_CHANGE'
DIAGNOSTIC_PCT_COL = 'NEXT_AREA_CHANGE_PCT'

# Temporal Split Boundaries (Current Observation Year)
TRAIN_MAX_YEAR = 2021
VAL_YEAR = 2022
TEST_YEAR = 2023
FINAL_OBS_YEAR = 2024  # No target available

# Prohibited Leakage Columns
PROHIBITED_LEAKAGE_COLUMNS = [
    'NEXT_AREA', 'NEXT_YEAR', 'NEXT_YEAR_GAP',
    'EXPANSION_RATE', 'EXPANSION_RATE_SIG', 'EXPANSION_UNCERTAINTY'
]

print(f'GLO directory    : {GLO_DIR}')
print(f'S1 Annual file   : {S1_ANNUAL_PATH.name}')
print(f'S2 Annual file   : {S2_ANNUAL_PATH.name}')
print(f'Target column    : {TARGET_COL}')
print(f'Train years      : <= {TRAIN_MAX_YEAR} (predicts up to 2022)')
print(f'Validation year  : {VAL_YEAR} (predicts 2023)')
print(f'Test year        : {TEST_YEAR} (predicts 2024)')


GLO directory    : C:\Users\ASUS\Desktop\GlacierGuard-AI\raw\GLO
S1 Annual file   : S1_20172024_NTB_GLOID_v1.02.gpkg
S2 Annual file   : S2_20172024_NTB_GLOID_v1.02.gpkg
Target column    : NEXT_AREA_CHANGE
Train years      : <= 2021 (predicts up to 2022)
Validation year  : 2022 (predicts 2023)
Test year        : 2023 (predicts 2024)


## 3 — Annual Data Ingestion & Base Validation

In [4]:
# == 3.1  Ingest Annual Datasets into PySpark ==
s1_pdf = gpd.read_file(str(S1_ANNUAL_PATH)).drop(columns='geometry')
s2_pdf = gpd.read_file(str(S2_ANNUAL_PATH)).drop(columns='geometry')

for col in ['START_DATE', 'END_DATE']:
    s1_pdf[col] = pd.to_datetime(s1_pdf[col])
    s2_pdf[col] = pd.to_datetime(s2_pdf[col])

s1_pdf['S2_MSTAT'] = s1_pdf['S2_MSTAT'].astype('Int32')
s1_pdf['AREA_YEAR'] = s1_pdf['AREA_YEAR'].astype(int)
s2_pdf['AREA_YEAR'] = s2_pdf['AREA_YEAR'].astype(int)

df_s1 = spark.createDataFrame(s1_pdf)
df_s2 = spark.createDataFrame(s2_pdf)

# Validation Assertions
s1_count = df_s1.count()
s2_count = df_s2.count()
assert s1_count == 18129, f'Expected 18,129 S1 rows, got {s1_count}'
assert s2_count == 22294, f'Expected 22,294 S2 rows, got {s2_count}'

# Verify zero duplicate (GLO_ID, AREA_YEAR) pairs
s1_dups = df_s1.groupBy('GLO_ID', 'AREA_YEAR').count().filter(F.col('count') > 1).count()
s2_dups = df_s2.groupBy('GLO_ID', 'AREA_YEAR').count().filter(F.col('count') > 1).count()
assert s1_dups == 0, f'Found {s1_dups} duplicate (GLO_ID, AREA_YEAR) pairs in S1'
assert s2_dups == 0, f'Found {s2_dups} duplicate (GLO_ID, AREA_YEAR) pairs in S2'

print(f'✓ Ingested S1: {s1_count:,} rows, {len(df_s1.columns)} columns (0 duplicates)')
print(f'✓ Ingested S2: {s2_count:,} rows, {len(df_s2.columns)} columns (0 duplicates)')


✓ Ingested S1: 18,129 rows, 20 columns (0 duplicates)
✓ Ingested S2: 22,294 rows, 19 columns (0 duplicates)


## 4 — Feature Contract Validation

In [5]:
# == 4.1  Explicit Feature Categorization ==
IDENTIFIER_COLS = ['GLO_ID']
TEMPORAL_COLS = ['AREA_YEAR']

CURRENT_OBS_COLS = [
    'AREA', 'PERIMETER', 'AREA_UNCERTAINTY',
    'ELEVATION_MEAN', 'ELEVATION_MIN', 'ELEVATION_MEDIAN',
    'CONNECTIVITY', 'LATITUDE', 'LONGITUDE',
    'BASIN', 'COUNTRY', 'GTNG_REGION_O2'
]

QC_COLS = ['TS_OUTLIER']

EXCLUDED_COLS = [
    'DATA_SOURCE', 'START_DATE', 'END_DATE', 'REF_MSTAT', 'S2_MSTAT',
    'EXPANSION_RATE', 'EXPANSION_RATE_SIG', 'EXPANSION_UNCERTAINTY'
]

print('Feature Contract Roles:')
print(f'  Identifiers         ({len(IDENTIFIER_COLS)}) : {IDENTIFIER_COLS}')
print(f'  Temporal            ({len(TEMPORAL_COLS)}) : {TEMPORAL_COLS}')
print(f'  Current Observation ({len(CURRENT_OBS_COLS)}) : {CURRENT_OBS_COLS}')
print(f'  Quality Control     ({len(QC_COLS)}) : {QC_COLS}')
print(f'  Explicit Exclusions ({len(EXCLUDED_COLS)}) : {EXCLUDED_COLS}')


Feature Contract Roles:
  Identifiers         (1) : ['GLO_ID']
  Temporal            (1) : ['AREA_YEAR']
  Current Observation (12) : ['AREA', 'PERIMETER', 'AREA_UNCERTAINTY', 'ELEVATION_MEAN', 'ELEVATION_MIN', 'ELEVATION_MEDIAN', 'CONNECTIVITY', 'LATITUDE', 'LONGITUDE', 'BASIN', 'COUNTRY', 'GTNG_REGION_O2']
  Quality Control     (1) : ['TS_OUTLIER']
  Explicit Exclusions (8) : ['DATA_SOURCE', 'START_DATE', 'END_DATE', 'REF_MSTAT', 'S2_MSTAT', 'EXPANSION_RATE', 'EXPANSION_RATE_SIG', 'EXPANSION_UNCERTAINTY']


## 5 — Temporal Transition & Gap Construction

Window specification partitioned by `GLO_ID` and ordered by `AREA_YEAR`:
- `PREV_YEAR`, `PREV_AREA`, `PREV_YEAR_GAP`
- `NEXT_YEAR`, `NEXT_AREA`, `NEXT_YEAR_GAP`
- Historical lookups: `PREV_YEAR_2`, `PREV_YEAR_3`, `PREV_YEAR_4`

In [6]:
# == 5.1  Window Function Construction ==
def add_temporal_transitions(df):
    w = Window.partitionBy('GLO_ID').orderBy('AREA_YEAR')
    
    return (df
        .withColumn('PREV_YEAR', F.lag('AREA_YEAR', 1).over(w))
        .withColumn('PREV_AREA', F.lag('AREA', 1).over(w))
        .withColumn('PREV_YEAR_GAP', F.col('AREA_YEAR') - F.col('PREV_YEAR'))
        .withColumn('PREV_YEAR_2', F.lag('AREA_YEAR', 2).over(w))
        .withColumn('PREV_AREA_2', F.lag('AREA', 2).over(w))
        .withColumn('PREV_YEAR_3', F.lag('AREA_YEAR', 3).over(w))
        .withColumn('PREV_AREA_3', F.lag('AREA', 3).over(w))
        .withColumn('PREV_YEAR_4', F.lag('AREA_YEAR', 4).over(w))
        .withColumn('PREV_AREA_4', F.lag('AREA', 4).over(w))
        .withColumn('NEXT_YEAR', F.lead('AREA_YEAR', 1).over(w))
        .withColumn('NEXT_AREA', F.lead('AREA', 1).over(w))
        .withColumn('NEXT_YEAR_GAP', F.col('NEXT_YEAR') - F.col('AREA_YEAR'))
    )

df_s1_trans = add_temporal_transitions(df_s1)
df_s2_trans = add_temporal_transitions(df_s2)

# Validation: ensure no non-positive gaps
s1_neg_gaps = df_s1_trans.filter(F.col('PREV_YEAR_GAP') <= 0).count()
s2_neg_gaps = df_s2_trans.filter(F.col('PREV_YEAR_GAP') <= 0).count()
assert s1_neg_gaps == 0, f'Found {s1_neg_gaps} invalid negative/zero gaps in S1'
assert s2_neg_gaps == 0, f'Found {s2_neg_gaps} invalid negative/zero gaps in S2'

print('✓ Temporal transition fields constructed without temporal ordering anomalies.')


✓ Temporal transition fields constructed without temporal ordering anomalies.


## 6 — Historical Lag Features (Strictly Backward-Looking)

Lags are populated **only** when all intervening annual observations are consecutive:
- `AREA_LAG1`: Valid only when $t - \text{PREV\_YEAR} == 1$.
- `AREA_LAG2`: Valid only when $t - \text{PREV\_YEAR} == 1$ AND $\text{PREV\_YEAR} - \text{PREV\_YEAR\_2} == 1$.
- `AREA_LAG3`: Valid only when all 3 preceding annual observations are strictly consecutive.
- `AREA_LAG4`: Valid only when all 4 preceding observations are strictly consecutive.

In [7]:
# == 6.1  Compute Gap-Enforced Historical Lags ==
def add_historical_lags(df):
    return (df
        # LAG1
        .withColumn(
            'AREA_LAG1',
            F.when(F.col('PREV_YEAR_GAP') == 1, F.col('PREV_AREA')).otherwise(F.lit(None))
        )
        # LAG2: Both transitions must be gap=1
        .withColumn(
            'AREA_LAG2',
            F.when(
                (F.col('PREV_YEAR_GAP') == 1) &
                ((F.col('PREV_YEAR') - F.col('PREV_YEAR_2')) == 1),
                F.col('PREV_AREA_2')
            ).otherwise(F.lit(None))
        )
        # LAG3: All three transitions must be gap=1
        .withColumn(
            'AREA_LAG3',
            F.when(
                (F.col('PREV_YEAR_GAP') == 1) &
                ((F.col('PREV_YEAR') - F.col('PREV_YEAR_2')) == 1) &
                ((F.col('PREV_YEAR_2') - F.col('PREV_YEAR_3')) == 1),
                F.col('PREV_AREA_3')
            ).otherwise(F.lit(None))
        )
        # LAG4: All four transitions must be gap=1
        .withColumn(
            'AREA_LAG4',
            F.when(
                (F.col('PREV_YEAR_GAP') == 1) &
                ((F.col('PREV_YEAR') - F.col('PREV_YEAR_2')) == 1) &
                ((F.col('PREV_YEAR_2') - F.col('PREV_YEAR_3')) == 1) &
                ((F.col('PREV_YEAR_3') - F.col('PREV_YEAR_4')) == 1),
                F.col('PREV_AREA_4')
            ).otherwise(F.lit(None))
        )
    )

df_s1_lags = add_historical_lags(df_s1_trans)
df_s2_lags = add_historical_lags(df_s2_trans)

# Assertion: When PREV_YEAR_GAP > 1, AREA_LAG1 must be NULL
s1_invalid_lag1 = df_s1_lags.filter((F.col('PREV_YEAR_GAP') > 1) & F.col('AREA_LAG1').isNotNull()).count()
s2_invalid_lag1 = df_s2_lags.filter((F.col('PREV_YEAR_GAP') > 1) & F.col('AREA_LAG1').isNotNull()).count()
assert s1_invalid_lag1 == 0, f'Found {s1_invalid_lag1} invalid AREA_LAG1 across gaps in S1'
assert s2_invalid_lag1 == 0, f'Found {s2_invalid_lag1} invalid AREA_LAG1 across gaps in S2'

print('✓ Historical area lags computed and gap-enforcement verified.')


✓ Historical area lags computed and gap-enforcement verified.


## 7 — Historical Area Changes

Compute backward consecutive differences:
- `AREA_CHANGE`: Current historical step ($A_t - A_{t-1}$ where `PREV_YEAR_GAP == 1`).
- `AREA_CHANGE_LAG1`: Prior step ($A_{t-1} - A_{t-2}$). Valid when both `AREA_LAG1` and `AREA_LAG2` exist.
- `AREA_CHANGE_LAG2`: Prior step ($A_{t-2} - A_{t-3}$). Valid when both `AREA_LAG2` and `AREA_LAG3` exist.
- `AREA_CHANGE_LAG3`: Prior step ($A_{t-3} - A_{t-4}$). Valid when both `AREA_LAG3` and `AREA_LAG4` exist.

In [8]:
# == 7.1  Compute Historical Area Changes ==
def add_historical_changes(df):
    return (df
        .withColumn(
            'AREA_CHANGE',
            F.when(F.col('PREV_YEAR_GAP') == 1, F.col('AREA') - F.col('PREV_AREA')).otherwise(F.lit(None))
        )
        .withColumn(
            'AREA_CHANGE_LAG1',
            F.when(
                F.col('AREA_LAG1').isNotNull() & F.col('AREA_LAG2').isNotNull(),
                F.col('AREA_LAG1') - F.col('AREA_LAG2')
            ).otherwise(F.lit(None))
        )
        .withColumn(
            'AREA_CHANGE_LAG2',
            F.when(
                F.col('AREA_LAG2').isNotNull() & F.col('AREA_LAG3').isNotNull(),
                F.col('AREA_LAG2') - F.col('AREA_LAG3')
            ).otherwise(F.lit(None))
        )
        .withColumn(
            'AREA_CHANGE_LAG3',
            F.when(
                F.col('AREA_LAG3').isNotNull() & F.col('AREA_LAG4').isNotNull(),
                F.col('AREA_LAG3') - F.col('AREA_LAG4')
            ).otherwise(F.lit(None))
        )
    )

df_s1_changes = add_historical_changes(df_s1_lags)
df_s2_changes = add_historical_changes(df_s2_lags)

print('✓ Historical area change lags constructed.')


✓ Historical area change lags constructed.


## 8 — Historical Rolling Features

Rolling statistics strictly use prior consecutive annual changes (`AREA_CHANGE_LAG1`, `AREA_CHANGE_LAG2`, `AREA_CHANGE_LAG3`):
- `ROLLING_MEAN_3`: Mean of available historical changes (min periods = 1).
- `ROLLING_STD_3`: Sample standard deviation of available historical changes (min periods = 2; `NULL` if <2 observations).
- Current transition and future transition (`NEXT_AREA_CHANGE`) are strictly excluded.

In [9]:
# == 8.1  Compute Backward-Looking Rolling Statistics ==
def add_rolling_features(df):
    c1 = F.col('AREA_CHANGE_LAG1')
    c2 = F.col('AREA_CHANGE_LAG2')
    c3 = F.col('AREA_CHANGE_LAG3')
    
    # Count valid historical changes
    n_valid = (
        F.when(c1.isNotNull(), 1).otherwise(0) +
        F.when(c2.isNotNull(), 1).otherwise(0) +
        F.when(c3.isNotNull(), 1).otherwise(0)
    )
    
    # Sum of valid historical changes
    sum_c = (
        F.coalesce(c1, F.lit(0.0)) +
        F.coalesce(c2, F.lit(0.0)) +
        F.coalesce(c3, F.lit(0.0))
    )
    
    mean_c = sum_c / n_valid
    
    # Sum of squared deviations for sample standard deviation
    sq_diff = (
        F.when(c1.isNotNull(), F.pow(c1 - mean_c, 2)).otherwise(F.lit(0.0)) +
        F.when(c2.isNotNull(), F.pow(c2 - mean_c, 2)).otherwise(F.lit(0.0)) +
        F.when(c3.isNotNull(), F.pow(c3 - mean_c, 2)).otherwise(F.lit(0.0))
    )
    
    return (df
        .withColumn(
            'ROLLING_MEAN_3',
            F.when(n_valid >= 1, mean_c).otherwise(F.lit(None))
        )
        .withColumn(
            'ROLLING_STD_3',
            F.when(n_valid >= 2, F.sqrt(sq_diff / (n_valid - 1))).otherwise(F.lit(None))
        )
    )

df_s1_rolling = add_rolling_features(df_s1_changes)
df_s2_rolling = add_rolling_features(df_s2_changes)

print('✓ Backward-looking rolling features computed.')


✓ Backward-looking rolling features computed.


## 9 — Target Construction & Diagnostic Relative Metric

- `NEXT_AREA_CHANGE = NEXT_AREA - AREA` (populated strictly when `NEXT_YEAR_GAP == 1`, else `NULL`).
- `NEXT_AREA_CHANGE_PCT = (NEXT_AREA_CHANGE / AREA) * 100` (diagnostic only).

In [10]:
# == 9.1  Construct Target and Diagnostic Metric ==
def add_target(df):
    return (df
        .withColumn(
            TARGET_COL,
            F.when(F.col('NEXT_YEAR_GAP') == 1, F.col('NEXT_AREA') - F.col('AREA')).otherwise(F.lit(None))
        )
        .withColumn(
            DIAGNOSTIC_PCT_COL,
            F.when(
                (F.col(TARGET_COL).isNotNull()) & (F.col('AREA') > 0),
                (F.col(TARGET_COL) / F.col('AREA')) * 100.0
            ).otherwise(F.lit(None))
        )
    )

df_s1_target = add_target(df_s1_rolling)
df_s2_target = add_target(df_s2_rolling)

# Assertion: TARGET must be NULL when NEXT_YEAR_GAP != 1
s1_inv_target = df_s1_target.filter((F.col('NEXT_YEAR_GAP') != 1) & F.col(TARGET_COL).isNotNull()).count()
s2_inv_target = df_s2_target.filter((F.col('NEXT_YEAR_GAP') != 1) & F.col(TARGET_COL).isNotNull()).count()
assert s1_inv_target == 0, f'Found {s1_inv_target} non-consecutive target rows in S1'
assert s2_inv_target == 0, f'Found {s2_inv_target} non-consecutive target rows in S2'

print('✓ Primary target constructed with strict 1-year transition enforcement.')


✓ Primary target constructed with strict 1-year transition enforcement.


## 10 — Quality Control (QC) Handling

Preserve `TS_OUTLIER` without row deletion. Create normalized `IS_TS_OUTLIER` indicator for sensitivity experiments in Notebook 04.

In [11]:
# == 10.1  QC Flag Normalization ==
def normalize_qc(df):
    return df.withColumn(
        'IS_TS_OUTLIER',
        F.when(F.upper(F.col('TS_OUTLIER').cast('string')).isin(['TRUE', '1']), True).otherwise(False)
    )

df_s1_qc = normalize_qc(df_s1_target)
df_s2_qc = normalize_qc(df_s2_target)

print('✓ Quality Control flag normalized.')


✓ Quality Control flag normalized.


## 11 & 12 — Assembly of Gold Feature Datasets (`gold_s1`, `gold_s2`)

Select final Gold modeling features. Drop temporary target transition and leakage fields (`NEXT_AREA`, `NEXT_YEAR`, `NEXT_YEAR_GAP`, `PREV_*`).

In [12]:
# == 11.1  Final Gold Modeling Feature Selection ==
GOLD_COLUMNS = [
    'GLO_ID',
    'AREA_YEAR',
    'AREA',
    'PERIMETER',
    'AREA_UNCERTAINTY',
    'ELEVATION_MEAN',
    'ELEVATION_MIN',
    'ELEVATION_MEDIAN',
    'CONNECTIVITY',
    'LATITUDE',
    'LONGITUDE',
    'BASIN',
    'COUNTRY',
    'GTNG_REGION_O2',
    'TS_OUTLIER',
    'IS_TS_OUTLIER',
    'AREA_LAG1',
    'AREA_LAG2',
    'AREA_LAG3',
    'AREA_CHANGE_LAG1',
    'AREA_CHANGE_LAG2',
    'AREA_CHANGE_LAG3',
    'ROLLING_MEAN_3',
    'ROLLING_STD_3',
    'NEXT_AREA_CHANGE'
]

gold_s1 = df_s1_qc.select(*GOLD_COLUMNS)
gold_s2 = df_s2_qc.select(*GOLD_COLUMNS)

print(f'gold_s1 schema finalized: {len(gold_s1.columns)} columns')
print(f'gold_s2 schema finalized: {len(gold_s2.columns)} columns')


gold_s1 schema finalized: 25 columns
gold_s2 schema finalized: 25 columns


## 13 — Data Quality Audit

In [13]:
# == 13.1  Data Quality Audit Execution ==
def audit_gold_dataset(df, label):
    total = df.count()
    valid_target = df.filter(F.col(TARGET_COL).isNotNull()).count()
    null_target = df.filter(F.col(TARGET_COL).isNull()).count()
    unique_lakes = df.select('GLO_ID').distinct().count()
    
    lag1_cnt = df.filter(F.col('AREA_LAG1').isNotNull()).count()
    lag2_cnt = df.filter(F.col('AREA_LAG2').isNotNull()).count()
    lag3_cnt = df.filter(F.col('AREA_LAG3').isNotNull()).count()
    
    chg_lag1_cnt = df.filter(F.col('AREA_CHANGE_LAG1').isNotNull()).count()
    chg_lag2_cnt = df.filter(F.col('AREA_CHANGE_LAG2').isNotNull()).count()
    chg_lag3_cnt = df.filter(F.col('AREA_CHANGE_LAG3').isNotNull()).count()
    
    roll_mean_cnt = df.filter(F.col('ROLLING_MEAN_3').isNotNull()).count()
    roll_std_cnt = df.filter(F.col('ROLLING_STD_3').isNotNull()).count()
    ts_outlier_cnt = df.filter(F.col('IS_TS_OUTLIER') == True).count()
    
    target_stats = df.filter(F.col(TARGET_COL).isNotNull()).agg(
        F.min(TARGET_COL).alias('min'),
        F.max(TARGET_COL).alias('max'),
        F.mean(TARGET_COL).alias('mean'),
        F.expr(f'percentile_approx({TARGET_COL}, 0.5)').alias('median'),
        F.stddev(TARGET_COL).alias('std')
    ).collect()[0]
    
    print(f'\n============================================================')
    print(f'{label} — DATA QUALITY AUDIT')
    print(f'============================================================')
    print(f'Total annual observations    : {total:,}')
    print(f'Unique glacial lakes         : {unique_lakes:,}')
    print(f'Valid next-year targets      : {valid_target:,} ({valid_target/total*100:.1f}%)')
    print(f'Unobserved/excluded targets  : {null_target:,} ({null_target/total*100:.1f}%)')
    print(f'Valid AREA_LAG1              : {lag1_cnt:,} ({lag1_cnt/total*100:.1f}%)')
    print(f'Valid AREA_LAG2              : {lag2_cnt:,} ({lag2_cnt/total*100:.1f}%)')
    print(f'Valid AREA_LAG3              : {lag3_cnt:,} ({lag3_cnt/total*100:.1f}%)')
    print(f'Valid AREA_CHANGE_LAG1       : {chg_lag1_cnt:,} ({chg_lag1_cnt/total*100:.1f}%)')
    print(f'Valid AREA_CHANGE_LAG2       : {chg_lag2_cnt:,} ({chg_lag2_cnt/total*100:.1f}%)')
    print(f'Valid AREA_CHANGE_LAG3       : {chg_lag3_cnt:,} ({chg_lag3_cnt/total*100:.1f}%)')
    print(f'ROLLING_MEAN_3 availability  : {roll_mean_cnt:,} ({roll_mean_cnt/total*100:.1f}%)')
    print(f'ROLLING_STD_3 availability   : {roll_std_cnt:,} ({roll_std_cnt/total*100:.1f}%)')
    print(f'TS_OUTLIER count             : {ts_outlier_cnt:,} ({ts_outlier_cnt/total*100:.1f}%)')
    print(f'Target stats (km²)           : min={target_stats["min"]:.6f}, max={target_stats["max"]:.6f}, mean={target_stats["mean"]:.6f}, median={target_stats["median"]:.6f}, std={target_stats["std"]:.6f}')

audit_gold_dataset(gold_s1, 'Sentinel-1')
audit_gold_dataset(gold_s2, 'Sentinel-2')



Sentinel-1 — DATA QUALITY AUDIT
Total annual observations    : 18,129
Unique glacial lakes         : 2,966
Valid next-year targets      : 14,242 (78.6%)
Unobserved/excluded targets  : 3,887 (21.4%)
Valid AREA_LAG1              : 14,242 (78.6%)
Valid AREA_LAG2              : 11,452 (63.2%)
Valid AREA_LAG3              : 9,114 (50.3%)
Valid AREA_CHANGE_LAG1       : 11,452 (63.2%)
Valid AREA_CHANGE_LAG2       : 9,114 (50.3%)
Valid AREA_CHANGE_LAG3       : 7,072 (39.0%)
ROLLING_MEAN_3 availability  : 11,452 (63.2%)
ROLLING_STD_3 availability   : 9,114 (50.3%)
TS_OUTLIER count             : 263 (1.5%)
Target stats (km²)           : min=-1.319218, max=1.462065, mean=0.000585, median=0.000087, std=0.032128

Sentinel-2 — DATA QUALITY AUDIT
Total annual observations    : 22,294
Unique glacial lakes         : 4,150
Valid next-year targets      : 16,610 (74.5%)
Unobserved/excluded targets  : 5,684 (25.5%)
Valid AREA_LAG1              : 16,610 (74.5%)
Valid AREA_LAG2              : 13,019 (58.4%)

## 14 — Chronological Temporal Split Audit

In [14]:
# == 14.1  Temporal Split Verification ==
def audit_temporal_splits(df, label):
    supervised = df.filter(F.col(TARGET_COL).isNotNull())
    
    train_df = supervised.filter(F.col('AREA_YEAR') <= TRAIN_MAX_YEAR)
    val_df = supervised.filter(F.col('AREA_YEAR') == VAL_YEAR)
    test_df = supervised.filter(F.col('AREA_YEAR') == TEST_YEAR)
    
    train_n = train_df.count()
    val_n = val_df.count()
    test_n = test_df.count()
    total_sup = supervised.count()
    
    # Integrity Assertions
    assert train_df.filter(F.col('AREA_YEAR') > TRAIN_MAX_YEAR).count() == 0, 'Train year violation'
    assert val_df.filter(F.col('AREA_YEAR') != VAL_YEAR).count() == 0, 'Val year violation'
    assert test_df.filter(F.col('AREA_YEAR') != TEST_YEAR).count() == 0, 'Test year violation'
    assert supervised.filter(F.col('AREA_YEAR') >= FINAL_OBS_YEAR).count() == 0, f'{FINAL_OBS_YEAR} found in supervised targets'
    assert (train_n + val_n + test_n) == total_sup, 'Split count mismatch'
    
    print(f'\n{label} Temporal Splits:')
    print(f'  Train      (<= {TRAIN_MAX_YEAR}) : {train_n:,} rows ({train_n/total_sup*100:.1f}%) [predicts 2018–2022]')
    print(f'  Validation (== {VAL_YEAR}) : {val_n:,} rows ({val_n/total_sup*100:.1f}%) [predicts 2023]')
    print(f'  Test       (== {TEST_YEAR}) : {test_n:,} rows ({test_n/total_sup*100:.1f}%) [predicts 2024]')
    print(f'  Total Supervised Targets   : {total_sup:,} rows')

audit_temporal_splits(gold_s1, 'Sentinel-1')
audit_temporal_splits(gold_s2, 'Sentinel-2')



Sentinel-1 Temporal Splits:
  Train      (<= 2021) : 10,161 rows (71.3%) [predicts 2018–2022]
  Validation (== 2022) : 2,026 rows (14.2%) [predicts 2023]
  Test       (== 2023) : 2,055 rows (14.4%) [predicts 2024]
  Total Supervised Targets   : 14,242 rows

Sentinel-2 Temporal Splits:
  Train      (<= 2021) : 11,858 rows (71.4%) [predicts 2018–2022]
  Validation (== 2022) : 2,245 rows (13.5%) [predicts 2023]
  Test       (== 2023) : 2,507 rows (15.1%) [predicts 2024]
  Total Supervised Targets   : 16,610 rows


## 15 — Leakage & Feature Independence Audit

In [15]:
# == 15.1  Programmatic Leakage Verification ==
for label, df in [('Sentinel-1', gold_s1), ('Sentinel-2', gold_s2)]:
    cols = set(df.columns)
    leaked = [c for c in PROHIBITED_LEAKAGE_COLUMNS if c in cols]
    assert len(leaked) == 0, f'LEAKAGE VIOLATION in {label}: Prohibited columns found: {leaked}'

print('✓ Leakage Audit Passed: Zero prohibited columns detected in gold feature vectors.')


✓ Leakage Audit Passed: Zero prohibited columns detected in gold feature vectors.


## 16 & 17 — Final Validation & Summary Report

In [16]:
# == 17.1  Final Pipeline Report ==
print('=' * 65)
print('GLACIERGUARD FEATURE ENGINEERING COMPLETE')
print('=' * 65)

summary_data = [
    ('Total Observations', f'{gold_s1.count():,}', f'{gold_s2.count():,}'),
    ('Unique Glacial Lakes', f'{gold_s1.select("GLO_ID").distinct().count():,}', f'{gold_s2.select("GLO_ID").distinct().count():,}'),
    ('Valid Supervised Targets', f'{gold_s1.filter(F.col(TARGET_COL).isNotNull()).count():,}', f'{gold_s2.filter(F.col(TARGET_COL).isNotNull()).count():,}'),
    ('Train Examples (<=2021)', f'{gold_s1.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") <= 2021)).count():,}', f'{gold_s2.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") <= 2021)).count():,}'),
    ('Val Examples (2022)', f'{gold_s1.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") == 2022)).count():,}', f'{gold_s2.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") == 2022)).count():,}'),
    ('Test Examples (2023)', f'{gold_s1.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") == 2023)).count():,}', f'{gold_s2.filter(F.col(TARGET_COL).isNotNull() & (F.col("AREA_YEAR") == 2023)).count():,}'),
    ('Total Gold Columns', f'{len(gold_s1.columns)}', f'{len(gold_s2.columns)}'),
]

sum_df = pd.DataFrame(summary_data, columns=['Metric', 'Sentinel-1 (SAR)', 'Sentinel-2 (Optical)'])
print(sum_df.to_string(index=False))


GLACIERGUARD FEATURE ENGINEERING COMPLETE
                  Metric Sentinel-1 (SAR) Sentinel-2 (Optical)
      Total Observations           18,129               22,294
    Unique Glacial Lakes            2,966                4,150
Valid Supervised Targets           14,242               16,610
 Train Examples (<=2021)           10,161               11,858
     Val Examples (2022)            2,026                2,245
    Test Examples (2023)            2,055                2,507
      Total Gold Columns               25                   25
